<a href="https://colab.research.google.com/github/betulbilhan2/ai-carbon-tracker/blob/main/NB04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04: Hesaplama Motoru (Kural Tabanlı Anlık Emisyon)

Bu notebook, **TerkenTech — Yapay Zeka Pipeline'ı (v4)** dokümanında belirtilen **4. Adım** işlemlerini gerçekleştirir.

**Görev:**
Kullanıcının mobilden anlık gönderdiği verileri, Makine Öğrenmesi (TabNet) kullanmadan, **Lookup (Referans) Tablolarına dayalı deterministik bir formülle** hesaplayarak kullanıcının *o haftaki (actual_weekly_kg)* gerçek emisyonunu bulmaktır.

Bu modül daha sonra `.py` dosyası olarak kaydedilip canlı API'ye entegre edilecektir.

In [ ]:
# Google Drive'ı bağlayalım
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

print("Drive bağlantısı başarılı!")

Mounted at /content/drive
Drive bağlantısı başarılı!


In [ ]:
# 1. Lookup Tablolarını Yükleme (Kütüphane Görevi Görecek Veri Setleri)
base_path = '/content/drive/MyDrive/terkentech'

df_food = pd.read_parquet(f'{base_path}/01_processed/lookup_tables/food_emission_lookup.parquet')
df_fuel = pd.read_parquet(f'{base_path}/01_processed/lookup_tables/fuel_emission_lookup.parquet')

print("Besin Emisyon Tablosu Boyutu:", df_food.shape)
print("Araç Yakıt Tablosu Boyutu:", df_fuel.shape)

# Araç ortalama yakıt tüketimi referansını bulalım (L/100km)
avg_fuel_consumption = df_fuel['Fuel Consumption (City (L/100 km)'].mean()
print(f"Araçların Ortalama Şehir İçi Yakıt Tüketimi: {avg_fuel_consumption:.2f} L / 100km")

# Besin türlerine göre temsili katsayılar (Carbon Emission per kg)
# (Tabloda et ürünleri genel olarak yüksektir, vegan düşüktür)
meat_emissions = df_food[df_food['Food product'].str.contains('Beef|Pork|Lamb|Poultry', case=False, na=False)]['Total_emissions'].mean()
vegan_emissions = df_food[df_food['Food product'].str.contains('Vegetables|Fruit|Wheat|Rice', case=False, na=False)]['Total_emissions'].mean()

# Eğer isim uyuşmazlığı yüzünden NaN dönerse, manuel fallback katsayılar ekleyelim (kg CO2 / kg besin):
if pd.isna(meat_emissions): meat_emissions = 15.0
if pd.isna(vegan_emissions): vegan_emissions = 2.0

print(f"Et ağırlıklı besin emisyon referansı: {meat_emissions:.2f}")
print(f"Vegan besin emisyon referansı: {vegan_emissions:.2f}")

Besin Emisyon Tablosu Boyutu: (43, 2)
Araç Yakıt Tablosu Boyutu: (946, 3)
Araçların Ortalama Şehir İçi Yakıt Tüketimi: 12.51 L / 100km
Et ağırlıklı besin emisyon referansı: 27.82
Vegan besin emisyon referansı: 1.20


In [ ]:
%%writefile /content/drive/MyDrive/terkentech/hesaplama_motoru.py
# 2. Python Modülünün Oluşturulması (hesaplama_motoru.py)
# Bu hücre çalıştırıldığında içindeki kodlar bir .py dosyasına kaydedilecektir.

class HesaplamaMotoru:
    def __init__(self, avg_fuel_consumption=10.0, meat_emission_factor=15.0, vegan_emission_factor=2.0):
        # Lookup verilerinden elde ettiğimiz ortalama katsayılar
        self.avg_fuel = avg_fuel_consumption
        self.meat_factor = meat_emission_factor
        self.vegan_factor = vegan_emission_factor
        self.co2_per_liter_petrol = 2.31 # kg CO2/Litre benzin
        self.co2_per_liter_diesel = 2.68 # kg CO2/Litre dizel

    def calculate_actual_weekly_kg(self, user_input):
        """
        API sözleşmesinden (Bölüm 5) gelen anlık veriyi kullanarak, kural tabanlı deterministik
        gerçek (actual) haftalık kg CO2 miktarını hesaplar.
        """
        total_weekly_co2 = 0.0

        # 1. Ulaşım Hesabı
        # Aylık mesafe haftalığa çevriliyor (Bölüm 5 Pipeline kuralı)
        weekly_km = user_input.get('vehicle_distance_km_month', 0) / 4.345
        vehicle_type = user_input.get('vehicle_type', 'none').lower()

        if vehicle_type == 'petrol':
            total_weekly_co2 += (weekly_km / 100) * self.avg_fuel * self.co2_per_liter_petrol
        elif vehicle_type == 'diesel':
            total_weekly_co2 += (weekly_km / 100) * self.avg_fuel * self.co2_per_liter_diesel
        elif vehicle_type == 'hybrid':
            total_weekly_co2 += (weekly_km / 100) * (self.avg_fuel * 0.6) * self.co2_per_liter_petrol
        # Elektrikli araçlar ve aracı olmayanlar (none/walk/bicycle) için doğrudan emisyon şimdilik 0 varsayılıyor

        # 2. Beslenme Hesabı (Ortalama bir insan haftada 14 kg yemek yer varsayımıyla)
        diet = user_input.get('diet', 'mixed').lower()
        if diet == 'vegan':
            total_weekly_co2 += 14 * self.vegan_factor
        elif diet == 'vegetarian' or diet == 'pescatarian':
            total_weekly_co2 += 14 * ((self.vegan_factor + self.meat_factor) / 3)
        else: # omnivore / mixed
            total_weekly_co2 += 14 * ((self.vegan_factor + self.meat_factor) / 2)

        # 3. Atık (Waste) Hesabı
        # Torba boyutu katsayısı: small=1, medium=2, large=3, extra_large=4
        bag_size_map = {'small': 1, 'medium': 2, 'large': 3, 'extra_large': 4}
        bag_size_str = user_input.get('waste_bag_size', 'medium').lower()
        bag_multiplier = bag_size_map.get(bag_size_str, 2)
        bag_count = user_input.get('waste_bag_weekly_count', 0)
        # Her 1 birim çöp hacmi için ortalama 1.5 kg CO2 atık emisyonu varsayımı
        total_weekly_co2 += (bag_count * bag_multiplier * 1.5)

        return round(total_weekly_co2, 2)


Writing /content/drive/MyDrive/terkentech/hesaplama_motoru.py


In [ ]:
# 3. Fonksiyonu Test Etme (API Sözleşmesi Örneği ile)

import sys
sys.path.append('/content/drive/MyDrive/terkentech')
from importlib import reload
import hesaplama_motoru  # Yukarıda kaydettiğimiz py dosyasını içe aktarıyoruz
reload(hesaplama_motoru)

motor = hesaplama_motoru.HesaplamaMotoru(
    avg_fuel_consumption=avg_fuel_consumption,
    meat_emission_factor=meat_emissions,
    vegan_emission_factor=vegan_emissions
)

# Bölüm 5'teki test verisi
test_user_input = {
  "user_id": "user_123",
  "transport": "car",
  "vehicle_type": "petrol",
  "vehicle_distance_km_month": 600,
  "diet": "mixed",
  "monthly_grocery_bill": 300,
  "heating_energy_source": "natural gas",
  "energy_efficiency": "medium",
  "waste_bag_size": "medium",
  "waste_bag_weekly_count": 3,
  "recycling": {"paper": True, "plastic": True, "glass": False, "metal": False},
  "air_travel_frequency": "rarely"
}

haftalik_kg = motor.calculate_actual_weekly_kg(test_user_input)

print("Kullanıcı API Girdisi:", test_user_input)
print(f"\nhesaplama_motoru.py çıktısı -> Gözlemlenen (Actual) Haftalık Karbon Emisyonu: {haftalik_kg} kg CO2")
print("\nTEBRİKLER! Notebook 4 tamamlandı ve modül .py olarak kaydedildi! 🚀")

Kullanıcı API Girdisi: {'user_id': 'user_123', 'transport': 'car', 'vehicle_type': 'petrol', 'vehicle_distance_km_month': 600, 'diet': 'mixed', 'monthly_grocery_bill': 300, 'heating_energy_source': 'natural gas', 'energy_efficiency': 'medium', 'waste_bag_size': 'medium', 'waste_bag_weekly_count': 3, 'recycling': {'paper': True, 'plastic': True, 'glass': False, 'metal': False}, 'air_travel_frequency': 'rarely'}

hesaplama_motoru.py çıktısı -> Gözlemlenen (Actual) Haftalık Karbon Emisyonu: 252.07 kg CO2

TEBRİKLER! Notebook 4 tamamlandı ve modül .py olarak kaydedildi! 🚀
